### Plotting setup

In [2]:
# basic imports
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import glob
import pandas as pd

# ============ for latex fonts ============
from matplotlib import rc  # , font_manager

rc(
    "text.latex", preamble=r"\usepackage{lmodern} \usepackage{physics}"
)  # this helps use the plots in tex files
plt.rcParams.update({"font.size": 11})
plt.rcParams.update(
    {
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "xtick.major.width": 0.8,
        "ytick.major.width": 0.8,
        "xtick.minor.width": 0.8,
        "ytick.minor.width": 0.8,
        "axes.labelsize": 9,
        "axes.titlesize": 10,
        "axes.linewidth": 0.8,
        "lines.linewidth": 0.8,
        "patch.linewidth": 0.8,
        "legend.fontsize": 8,
        "legend.title_fontsize": 9,
        "legend.fancybox": False,
        "legend.frameon": False,
        "legend.handlelength": 1.0,
        # "legend.handletextpad": 0.5,
        "legend.labelspacing": 0.4,
        "figure.titlesize": 12,
        "figure.figsize": [3.54, 3.0],
        "figure.dpi": 300,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
        "mathtext.fontset": "cm",
        "text.usetex": True,
        "font.family": "Computer Modern Roman",
    }
)
# ==========================================

###################

data_dir = Path("build/data")
plot_dir = Path("plot")

if not Path.exists(plot_dir):
    Path.mkdir(plot_dir)

###################


def load_data(filename, data_dir=data_dir):
    data = np.loadtxt(data_dir / filename)
    return (col for col in data.T)


def prep_image_data(x, y, z):
    x = np.unique(x)
    y = np.unique(y)

    X, Y = np.meshgrid(x, y)
    Z = z.reshape(len(x), len(y))

    return (np.transpose(Z), [min(x), max(x), min(y), max(y)])
    # data = pd.DataFrame({"x": x, "y": y, "z": z})
    # data_pivot = data.pivot_table(index="y", columns="x", values="z")
    # extent = [min(x), max(x), min(y), max(y)]
    # return data_pivot, extent


def set_labels(x=r"$x$", y=r"$y$", title=""):
    plt.xlabel(x)
    plt.ylabel(y)
    plt.title(title)


def save_plot(filename, dpi=300, bbox_inches="tight", plot_dir=plot_dir):
    plt.tight_layout()
    plt.savefig(plot_dir / filename, bbox_inches=bbox_inches, dpi=dpi)
    plt.close("all")


##########################

def symlog10(x):
    return np.sign(x) * np.log10((np.abs(x) + 1.0))

### slice from 2D data

In [ ]:
def get_vertical_slice(x, y, z, axis, offset=0.0):
    ux = np.unique(x)
    uy = np.unique(y)

    match axis:
        case "x":
            diff = np.abs(ux - offset)
            val = ux[np.argmin(diff)]

            return uy, z[x == val]

        case "y":
            diff = np.abs(uy - offset)
            val = uy[np.argmin(diff)]

            return ux, z[y == val]

        case _:
            raise ValueError("Invalid axis")


# something

### Band structure slice

In [247]:
k, *energies = load_data(f"BS_slice.dat")
for band_idx, E in enumerate(energies):
    plt.plot(k, E)

# plt.xlim(-0.6, 0.6)
# plt.ylim(-0.5, 0.5)
plt.ylim(-20, 20)
# plt.ylim(-50, 50)
set_labels(r"$k_x\ (1/a)$ ", r"$E$ (meV)", r"$\mu = 0.0$ meV, $k_y = 0.0$")
plt.grid()
save_plot("BS_slice.png")

### Band structure discrete ky

In [246]:
kx, *energies = load_data(f"BS.dat")
for band_idx, E in enumerate(energies):
    plt.plot(kx, E)

# plt.xlim(-0.6, 0.6)
# plt.ylim(-0.5, 0.5)
plt.ylim(-2, 10)
# plt.ylim(-0.5, 0.5)
set_labels(r"$k_x\ (1/a)$ ", r"$E$ (meV)", r"$n_{k_y} = 21, B_z=1\ \text{T}, \mu = 0.0$ meV")
plt.grid()
save_plot("BS.png")

### Discrete hamiltonian Energy vs parameter

In [245]:
p, *energies = load_data(f"energy.dat")
for band_idx, E in enumerate(energies):
    plt.scatter(p, E, s=0.2, c="k")
    
# plt.xlim(0, 1)
set_labels(r"$B_z$ (T)", r"$E$ (meV)")
plt.grid(alpha=0.3)
save_plot("energies.png")

FileNotFoundError: build/data/energy.dat not found.

### Discrete system prob density

In [170]:
x,y,pd = load_data(f"prob_den.dat")

pd, extent = prep_image_data(x, y, pd)
plt.imshow(pd, extent=extent, aspect="auto", cmap="Reds")
plt.colorbar()
plt.gca().set_box_aspect(max(y) / max(x))
set_labels(r"$x\ (a)$", r"$y\ (a)$", r"$|\psi|^2$")
save_plot("prob_den.png")
sum(pd.flatten())

np.float64(1.0000000991356002)

### Band structure 3D

In [ ]:
kx, ky, *energies = load_data("BS.dat")
ukx, uky = np.unique(kx), np.unique(ky)
x, y = np.meshgrid(ukx, uky)

fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(111, projection="3d")

for i, e in enumerate(energies):
    ax.plot_surface(x, y, e.reshape(len(ukx), len(uky)).T, alpha=0.5, label=f"{i}")

set_labels(r"$k_x$", r"$k_y$", r"$E$ (meV)")
ax.legend(title="Band index")
save_plot("bands_3d.png")

### Gap closing with changes in mu

In [ ]:
fig, axes = plt.subplots(3, 3, sharex=True, sharey=True, figsize=(5, 5.5))

mu_start, mu_end = -1.0, 1.0

for i, mu in enumerate(np.linspace(mu_start, mu_end, 9)):
    k, *energies = load_data(f"BS{i}.dat")

    for band_idx, E in enumerate(energies):
        axes.flat[i].plot(k, E)

    axes.flat[i].set_title(r"$\mu = " + f"{mu:.2f}"+r"$ meV")

for ax in axes.flat:
    # ax.set_xlim(-1, 1)
    # ax.set_xlim(-0.15, 0.15)

    ax.set_ylim(-2, 2)
    # ax.set_ylim(-4, 0)
    ax.set_xlabel(r"$k_x$ ($1/a$)")
    ax.set_ylabel(r"$E$ (meV)")
    ax.set_box_aspect(1)
    ax.label_outer()
    ax.grid()

save_plot(f"gap_closing.png", dpi=600)

### Chern numbers vs mu

In [244]:
# n_bands = 4*9
# n_dense, n_sparse = 1000, 500

# # n_bands = 12
# # n_dense, n_sparse = 5000, 1500

# mu, *Cs = load_data("mu_Cs.dat")

# for i, C in enumerate(Cs[: (n_bands // 2)]):
#     plt.plot(mu, C, label=f"{i}")

# plt.plot(mu,  np.mod(np.sum(Cs[: (n_bands // 2)], axis=0),1), label="Sum")

# plt.grid(alpha=0.5)
# plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", title="Band")
# set_labels(r"$\mu$ (meV)", r"$C$", f"{n_dense=}, {n_sparse=}")

# save_plot(f"mu_Cs_{n_dense}_{n_sparse}.png")


n_dense, n_sparse = 10, 10

p, CN = load_data("CN.dat")
plt.plot(p, CN)

plt.grid(alpha=0.5)
# set_labels(r"$\mu$ (meV)", r"$C$", f"{n_dense=}, {n_sparse=}")
set_labels(r"$B_z$ (T)", r"$C$", f"{n_dense=}, {n_sparse=}")

save_plot(f"CN_{n_dense}_{n_sparse}.png")

### Chern numbers vs mu and Bz grid

In [236]:
# n_bands = 12

# # mu, Bz, *Cs = load_data("mu_Bz_Cs.dat")
# mu, Bz, *Cs = np.loadtxt("LAOSTO/mu_Bz_Cs_4000_1000.dat", unpack=True)

# C = np.sum(Cs[: (n_bands // 2)], axis=0)

# Z, e = prep_image_data(mu, Bz, C)
# plt.imshow(Z, extent=e, origin="lower", aspect="auto", cmap="RdBu", vmin=-1, vmax=1)

# plt.colorbar()
# set_labels(r"$\mu$ (meV)", r"$B_z$ (T)")

# save_plot("mu_Bz_Cs.png")



# for i, C in enumerate(Cs[: (n_bands // 2)]):
#     Z, e = prep_image_data(mu, Bz, C)
#     # Z[Z.shape[0] // 2-1].fill(np.nan)
#     # Z[Z.shape[0] // 2].fill(np.nan)
#     # Z[Z.shape[0] // 2+1].fill(np.nan)
#     plt.imshow(Z, extent=e, origin="lower", aspect="auto", cmap="RdBu")
#     plt.colorbar()
#     set_labels(r"$\mu$ (meV)", r"$B_z$ (T)", f"Band {i}")
#     save_plot(f"mu_Bz_C_{i}.png")

mu, Bz, CN = load_data("mu_Bz_CN.dat")

Z, e = prep_image_data(mu, Bz, CN)
plt.imshow(Z, extent=e, origin="lower", aspect="auto", cmap="Blues")

plt.colorbar()
set_labels(r"$\mu$ (meV)", r"$B_z$ (T)", r"$n_{k_y} = 21$") 

save_plot("mu_Bz_CN.png")

### Berry Curvature

In [71]:
kx, ky, *BCs = load_data("BC.dat")
fig, axes = plt.subplots(3, 4, sharex=True, sharey=True, figsize=(10, 7))

for iBC, BC in enumerate(BCs):
    # BC = symlog10(BC)
    Z, e = prep_image_data(kx, ky, BC)
    cmax = np.abs(BC).max()
    mapped = axes.flat[iBC].imshow(
        Z, extent=e, origin="lower", aspect="equal", cmap="RdBu", vmin=-cmax, vmax=cmax
    )
    axes.flat[iBC].set_xlabel(r"$k_x$")
    axes.flat[iBC].set_ylabel(r"$k_y$")
    axes.flat[iBC].label_outer()
    axes.flat[iBC].set_title(f"Band {iBC}")
    axes.flat[iBC].set_box_aspect(1)
    fig.colorbar(mapped, ax=axes.flat[iBC])

fig.suptitle(r"SymLog10 of Berry Curvature for $\mu = 0.0$ meV")
save_plot(f"BC.png", dpi=600)

### Gap vs momentum

In [80]:
kx, ky, *gaps = load_data("gap.dat")
gap = gaps[0]

Z, e = prep_image_data(kx, ky, gap)
plt.imshow(Z, extent=e, origin="lower", aspect="equal", cmap="jet")

plt.colorbar()
set_labels(r"$k_x$", r"$k_y$", r"Gap (meV)")

save_plot("gap.png")

np.min(gap), np.max(gap)

(np.float64(0.2), np.float64(3.00308))

### FS contours

In [10]:
for i, contour_file in enumerate(glob.glob("build/data/gap_FS*.dat")):
    kx, ky, *_ = np.loadtxt(contour_file, unpack=True)
    plt.plot(kx, ky, label=f"{i}")
    fs = (kx, ky)

plt.xlim(-0.5, 0.5)
plt.ylim(-0.5, 0.5)
plt.gca().set_box_aspect(1)
set_labels(r"$k_x$", r"$k_y$", r"FS")

save_plot("FS.png")

### FS contours and gap along them

In [79]:
for i, contour_file in enumerate(glob.glob("build/data/gap_FS*.dat")):
    kx, ky, *_ = np.loadtxt(contour_file, unpack=True)
    plt.plot(kx, ky, label=f"{i}")

plt.xlim(-0.5, 0.5)
plt.ylim(-0.5, 0.5)
# plt.xlim(-1.2, 1.2)
# plt.ylim(-1.2, 1.2)
plt.gca().set_box_aspect(1)
set_labels(r"$k_x$", r"$k_y$", r"FS")

save_plot("FS.png")

n_bands = 4

for ib in range(n_bands):
    for i, contour_file in enumerate(glob.glob("build/data/gap_FS*.dat")):
        kx, ky, *gaps = np.loadtxt(contour_file, unpack=True)
        theta = np.arctan2(ky[1:], kx[1:])
        gap = gaps[ib][1:]
        plt.plot(theta, gap, label=f"{i}")

    plt.xlim(-np.pi, np.pi)
    # plt.ylim(0.0, 0.05)
    plt.xticks(
        np.linspace(-np.pi, np.pi, 5),
        [r"$-\pi$", r"$-\pi/2$", r"$0$", r"$\pi/2$", r"$\pi$"],
    )
    # plt.legend(title="contour idx")
    # plt.yscale("log")
    set_labels(r"$\theta$", r"$\Delta$", r"Gap along FS contours")

    save_plot(f"gap_along_FS_{ib}.png")

# #free up variables
# del  gaps, contour_file, kx, ky, theta, gap

### FS countours colored according to gap - long plotting time


In [ ]:
kx_ky_gap_lst = []
for i, contour_file in enumerate(glob.glob("build/data/gap_FS*.dat")):
    kx, ky, *gaps = np.loadtxt(contour_file, unpack=True)
    kx_ky_gap_lst.append((kx, ky, gaps[0]))

import matplotlib.cm as cm
import matplotlib.colors as mcolors

all_g = np.concatenate([g for _, _, g in kx_ky_gap_lst])
norm = mcolors.Normalize(vmin=all_g.min(), vmax=all_g.max())
cmap = cm.jet  # Colormap

for x, y, z in kx_ky_gap_lst:
    colors = cmap(norm(z))  # Map z-values to colors
    for i in range(len(x) - 1):
        plt.plot(x[i : i + 2], y[i : i + 2], color=colors[i])

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
plt.colorbar(sm, ax=plt.gca())

set_labels(r"$k_x$", r"$k_y$", r"Gap (meV)")

save_plot("FS_gap_colored.png")

### Abs delta

In [81]:
kx, ky, *abs_deltas = load_data("abs_delta.dat")

lst = []

for i, abs_delta in enumerate(abs_deltas):
    Z,e = prep_image_data(kx, ky, abs_delta)
    plt.imshow(Z, extent=e, origin="lower", aspect="equal", cmap="jet")
    # plt.imshow(np.sqrt(np.abs(Z)), extent=e, origin="lower", aspect="equal", cmap="jet")
    set_labels(r"$k_x$", r"$k_y$", r"$\Delta^2$ (meV)")
    plt.colorbar() 
    save_plot(f"abs_delta_{i}.png")
    
# # plot sum of abs_deltas
# # Z,e = prep_image_data(kx, ky, np.sum(list(map(np.abs, abs_deltas)), axis=0))
# Z,e = prep_image_data(kx, ky, np.sum(abs_deltas, axis=0))
# plt.imshow(Z, extent=e, origin="lower", aspect="equal", cmap="jet")
# set_labels(r"$k_x$", r"$k_y$", r"$\sum\Delta^2$ (meV)")
# plt.colorbar()
# save_plot(f"abs_delta_sum.png")

FileNotFoundError: build/data/abs_delta.dat not found.

### Abs Q matrix

In [8]:
kx, ky, *abs_deltas = load_data("delta.dat")

n_bands = 2

fig, axes = plt.subplots(n_bands, n_bands, sharex=True, sharey=True, figsize=(n_bands*2, n_bands*2))
for i, ax in enumerate(axes.flat):
    Z, e = prep_image_data(kx, ky, abs_deltas[i])

    mapped = ax.imshow(
        np.log10(Z), extent=e, origin="lower", aspect="equal", cmap="Reds", 
    )
    
    ax.set_xlabel(r"$k_x$")
    ax.set_ylabel(r"$k_y$")
    ax.label_outer()
    ax.set_box_aspect(1)
    fig.colorbar(mapped, ax=ax)
    
    print(f"Band {i}: min = {np.min(Z)}, max = {np.max(Z)}")

save_plot(f"abs_deltas.png", dpi=600)

Band 0: min = 0.2, max = 1.8578
Band 1: min = 0.217064, max = 7.81381
Band 2: min = 1.67223e-05, max = 7.37968
Band 3: min = 0.2, max = 1.8578


### q matrix determinant and trace

In [9]:
kx, ky, det_re, det_im, tr_re, tr_im = load_data("DT.dat")

fig, axes = plt.subplots(2, 2, sharex=True, sharey=True, figsize=(5, 5))

data = [det_re, det_im, np.log10(np.sqrt(det_re**2 + det_im**2)), np.atan2(det_im, det_re)]
title = ["Det Re", "Det Im", "Det Abs Log10", "Det Arg"] 

np.sqrt(det_re**2 + det_im**2)

for i, ax in enumerate(axes.flat):
    Z, e = prep_image_data(kx, ky, data[i])

    mapped = ax.imshow(
        Z, extent=e, origin="lower", aspect="equal", cmap="Reds", 
    )
    
    ax.set_xlabel(r"$k_x$")
    ax.set_ylabel(r"$k_y$")
    ax.set_title(title[i])
    ax.label_outer()
    ax.set_box_aspect(1)
    fig.colorbar(mapped, ax=ax)
    
    print(f"{title[i]}: min = {np.min(data[i])}, max = {np.max(data[i])}")
    

save_plot(f"det.png", dpi=300)

Det Re: min = -0.216939, max = 54.2921
Det Im: min = -0.738801, max = 0.738801
Det Abs Log10: min = -4.034572505728551, max = 1.734776846798161
Det Arg: min = -3.141592653589788, max = 2.7454733498470696
